In [1]:
import re
import pandas as pd
import ollama
from sqlalchemy import text
from pipeline_utils import engine

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# Customer name resolution
def resolve_customer_by_name(name_query, max_display=6):
    query = text("""
        SELECT DISTINCT `Customer ID`, `Customer Name`, `Customer Email`
        FROM tickets
        WHERE `Customer Name` LIKE :pattern
    """)
    matches = pd.read_sql(query, con=engine, params={"pattern": f"%{name_query}%"})

    if len(matches) == 0:
        return {"status": "no_match"}
    elif len(matches) == 1:
        return {"status": "single_match", "customer": matches.iloc[0].to_dict()}
    elif len(matches) <= max_display:
        return {"status": "ambiguous", "candidates": matches.to_dict("records")}
    else:
        return {"status": "too_many", "count": len(matches)}

In [3]:
def format_disambiguation_message(resolution_result, name_query):
    if resolution_result["status"] == "no_match":
        return f"I couldn't find any customer matching \"{name_query}\". Could you check the spelling or try their email instead?"

    elif resolution_result["status"] == "too_many":
        return (f"There are {resolution_result['count']} customers matching \"{name_query}\" — "
                f"that's too many to list. Could you narrow it down with a full name, "
                f"email address, or a product they mentioned?")

    elif resolution_result["status"] == "ambiguous":
        lines = [f"I found a few customers matching \"{name_query}\":"]
        for i, c in enumerate(resolution_result["candidates"], 1):
            lines.append(f"{i}. {c['Customer Name']} ({c['Customer Email']})")
        lines.append("Which one did you mean? You can reply with a number or their email.")
        return "\n".join(lines)

In [4]:
print(format_disambiguation_message(resolve_customer_by_name("Allison"), "Allison"))
print()
print(format_disambiguation_message(resolve_customer_by_name("Allison King"), "Allison King"))
print()
print(resolve_customer_by_name("Allison Sullivan"))

There are 25 customers matching "Allison" — that's too many to list. Could you narrow it down with a full name, email address, or a product they mentioned?

I found a few customers matching "Allison King":
1. Allison King (jensenwilliam@example.org)
2. Allison King (millerchristine@example.net)
Which one did you mean? You can reply with a number or their email.

{'status': 'single_match', 'customer': {'Customer ID': 'C2024', 'Customer Name': 'Allison Sullivan', 'Customer Email': 'davidford@example.net'}}


In [5]:
# Once the agent is shown a numbered shortlist, 
# this resolves their follow-up reply (a number or an email) back to a specific customer.
def resolve_disambiguation_reply(reply, candidates):
    reply = reply.strip().lower()

    if reply.isdigit():
        index = int(reply) - 1
        if 0 <= index < len(candidates):
            return candidates[index]
        return None

    for c in candidates:
        if c["Customer Email"].lower() == reply:
            return c

    for c in candidates:
        if reply in c["Customer Email"].lower():
            return c

    return None

In [6]:
candidates = resolve_customer_by_name("Allison King")["candidates"]

print(resolve_disambiguation_reply("1", candidates))
print(resolve_disambiguation_reply("millerchristine@example.net", candidates))
print(resolve_disambiguation_reply("jensenwilliam", candidates))  # partial email
print(resolve_disambiguation_reply("3", candidates))  # invalid, should be None

{'Customer ID': 'C0224', 'Customer Name': 'Allison King', 'Customer Email': 'jensenwilliam@example.org'}
{'Customer ID': 'C1897', 'Customer Name': 'Allison King', 'Customer Email': 'millerchristine@example.net'}
{'Customer ID': 'C0224', 'Customer Name': 'Allison King', 'Customer Email': 'jensenwilliam@example.org'}
None


In [ ]:
# extracts a ticket ID from the agent's message, if present. Returns None if no valid ticket ID is found.
def extract_ticket_id(user_message):
    match = re.search(r'\b(\d{1,6})\b', user_message)
    if match:
        return int(match.group(1))
    return None

In [8]:
print(extract_ticket_id("Show me ticket 4491"))
print(extract_ticket_id("Can you check on ticket number 7652"))
print(extract_ticket_id("What about ticket #123"))
print(extract_ticket_id("Tell me about this customer's issue"))  # no number -> should be None

4491
7652
123
None


In [9]:
# One LLM call handles both: classifies the message as `ticket_id` / `customer_name` / `unclear`, and when 
# it's a name lookup, extracts just the name (not the full sentence) so it can be passed straight to 
# `resolve_customer_by_name`

def classify_intent(user_message, awaiting_disambiguation=False):
    if awaiting_disambiguation:
        return {"intent": "disambiguation_reply", "name": None}

    prompt = f"""Analyze this customer support agent's message.

Message: "{user_message}"

Determine the intent - exactly one of:
- ticket_id: the message references a specific ticket number
- customer_name: the message asks about a specific customer by name
- unclear: neither applies clearly

If the intent is customer_name, also extract just the customer's name as written (no extra words).

Respond in EXACTLY this format, nothing else:
Intent: <ticket_id, customer_name, or unclear>
Name: <the customer's name, or None if not applicable>"""

    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    result = response["message"]["content"].strip()

    intent = "unclear"
    name = None
    for line in result.splitlines():
        if line.lower().startswith("intent:"):
            value = line.split(":", 1)[1].strip().lower()
            if "ticket_id" in value:
                intent = "ticket_id"
            elif "customer_name" in value:
                intent = "customer_name"
        elif line.lower().startswith("name:"):
            value = line.split(":", 1)[1].strip()
            if value.lower() != "none":
                name = value

    return {"intent": intent, "name": name}

In [10]:
print(classify_intent("What did Allison order?"))
print(classify_intent("Show me ticket 4491"))
print(classify_intent("Has Michael Allison complained about anything recently?"))

{'intent': 'customer_name', 'name': 'Allison'}
{'intent': 'ticket_id', 'name': None}
{'intent': 'customer_name', 'name': 'Michael Allison'}


In [11]:
# Single entry point for the chatbot
# given the agent's message and session state, returns what should happen next (resolved ticket, resolved 
# customer, a disambiguation prompt to show, or a clarifying reply).

def route_chat_message(user_message, session_state):

    # If we're mid-disambiguation, treat this message as a reply, not a new query
    if session_state.get("awaiting_disambiguation"):
        candidates = session_state["disambiguation_candidates"]
        resolved = resolve_disambiguation_reply(user_message, candidates)

        if resolved is None:
            return {
                "action": "reply",
                "message": "I didn't recognize that choice. Please reply with a number from the list, or an email address."
            }

        session_state["awaiting_disambiguation"] = False
        session_state["disambiguation_candidates"] = None
        return {
            "action": "customer_resolved",
            "customer_id": resolved["Customer ID"],
            "customer_name": resolved["Customer Name"]
        }

    # Otherwise, classify intent fresh
    intent_result = classify_intent(user_message)
    intent = intent_result["intent"]

    if intent == "ticket_id":
        ticket_id = extract_ticket_id(user_message)
        if ticket_id is None:
            return {"action": "reply", "message": "I couldn't find a ticket number in that message. Could you specify the ticket ID?"}
        return {"action": "ticket_resolved", "ticket_id": ticket_id}

    elif intent == "customer_name":
        name_query = intent_result["name"] or user_message  # fallback if extraction failed
        resolution = resolve_customer_by_name(name_query)

        if resolution["status"] == "single_match":
            c = resolution["customer"]
            return {"action": "customer_resolved", "customer_id": c["Customer ID"], "customer_name": c["Customer Name"]}

        elif resolution["status"] == "ambiguous":
            session_state["awaiting_disambiguation"] = True
            session_state["disambiguation_candidates"] = resolution["candidates"]
            return {"action": "reply", "message": format_disambiguation_message(resolution, name_query)}

        else:
            return {"action": "reply", "message": format_disambiguation_message(resolution, name_query)}

    else:
        return {"action": "reply", "message": "I'm not sure if you're asking about a ticket number or a customer name. Could you rephrase — e.g. 'show me ticket 4491' or 'what did Allison order'?"}

In [12]:
session_state = {}

print(route_chat_message("Show me ticket 4491", session_state))
print(route_chat_message("What did Allison King order?", session_state))
print("awaiting_disambiguation:", session_state.get("awaiting_disambiguation"))
print(route_chat_message("1", session_state))
print("awaiting_disambiguation:", session_state.get("awaiting_disambiguation"))

{'action': 'ticket_resolved', 'ticket_id': 4491}
{'action': 'reply', 'message': 'I found a few customers matching "Allison King":\n1. Allison King (jensenwilliam@example.org)\n2. Allison King (millerchristine@example.net)\nWhich one did you mean? You can reply with a number or their email.'}
awaiting_disambiguation: True
{'action': 'customer_resolved', 'customer_id': 'C0224', 'customer_name': 'Allison King'}
awaiting_disambiguation: False
